In [2]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from transformers import AutoTokenizer, AutoModel
from torch.utils.data import DataLoader, TensorDataset


In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_name = "dccuchile/bert-base-spanish-wwm-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
bert_model = AutoModel.from_pretrained(model_name).to(device)

Some weights of BertModel were not initialized from the model checkpoint at dccuchile/bert-base-spanish-wwm-uncased and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [4]:
def get_bert_embeddings(texts):
    bert_model.eval()
    embeddings = []
    with torch.no_grad():
        for text in texts:
            inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=128).to(device)
            outputs = bert_model(**inputs)
            # Usamos el token [CLS]
            embeddings.append(outputs.last_hidden_state[:, 0, :].cpu().numpy())
    return np.vstack(embeddings)

In [5]:
class VAE(nn.Module):
    def __init__(self, input_dim=768, latent_dim=32):
        super(VAE, self).__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 256), nn.ReLU(),
            nn.Linear(256, 128), nn.ReLU()
        )
        self.fc_mu = nn.Linear(128, latent_dim)
        self.fc_logvar = nn.Linear(128, latent_dim)
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128), nn.ReLU(),
            nn.Linear(128, 256), nn.ReLU(),
            nn.Linear(256, input_dim)
        )

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x):
        h = self.encoder(x)
        mu, logvar = self.fc_mu(h), self.fc_logvar(h)
        z = self.reparameterize(mu, logvar)
        return self.decoder(z), mu, logvar

In [6]:
def cargar_configuracion_aumento(file_path):
    """Lee el TXT y devuelve un diccionario { 'F20': 0.0, ... }"""
    config = {}
    with open(file_path, 'r') as f:
        for line in f:
            if '|' in line:
                diag, valor = line.strip().split('|')
                config[diag] = float(valor)
    return config

In [15]:
def pipeline_vae_personalizado(csv_path, config_path):
    # 1. Cargar datos y configuración
    df = pd.read_csv(csv_path, sep='|')
    df['Descripcion_Texto'] = df.iloc[:, :-1].apply(lambda row: ' '.join([str(x) for x in row if pd.notna(x) and x.strip() != '']), axis=1)
    config_aumento = cargar_configuracion_aumento(config_path)
    
    lista_embeddings_final = []
    
    # 2. Iterar por cada diagnóstico presente en el dataset
    for diag in df['DIAG PSQ'].unique():
        subset = df[df['DIAG PSQ'] == diag]
        
        # Obtener porcentaje del TXT (si no está, usamos 0.0 por seguridad)
        porcentaje = config_aumento.get(diag, 0.0)
        
        # --- PARTE A: PROCESAR REALES ---
        print(f"\n>>> Procesando {diag} (Reales: {len(subset)})")
        embeddings_reales = get_bert_embeddings(subset['Descripcion_Texto'].tolist())
        
        # Guardamos los reales primero
        df_real = pd.DataFrame(embeddings_reales)
        df_real['DIAG PSQ'] = diag
        df_real['Origen'] = 'Real'
        lista_embeddings_final.append(df_real)
        
        # --- PARTE B: GENERAR SINTÉTICOS SI CORRESPONDE ---
        n_generar = int(len(subset) * porcentaje)
        
        if n_generar > 0:
            print(f"    Generando {n_generar} sintéticos (Aumento del {porcentaje*100}%)...")
            
            # Entrenar VAE para esta clase específica
            vae = VAE().to(device)
            optimizer = torch.optim.Adam(vae.parameters(), lr=1e-3)
            X_train = torch.FloatTensor(embeddings_reales).to(device)
            
            vae.train()
            for _ in range(50): # Entrenamiento rápido
                optimizer.zero_grad()
                recon, mu, logvar = vae(X_train)
                loss = nn.functional.mse_loss(recon, X_train, reduction='sum') + \
                       (-0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()))
                loss.backward()
                optimizer.step()
            
            # Muestrear nuevos vectores
            vae.eval()
            with torch.no_grad():
                z = torch.randn(n_generar, 32).to(device)
                sinteticos = vae.decoder(z).cpu().numpy()
            
            # Guardar sintéticos
            df_sint = pd.DataFrame(sinteticos)
            df_sint['DIAG PSQ'] = diag
            df_sint['Origen'] = 'Sintetico'
            lista_embeddings_final.append(df_sint)
        else:
            print(f"    Saltando aumento para {diag} (Porcentaje 0).")

    # Unir todo en un solo dataframe numérico
    df_final = pd.concat(lista_embeddings_final).reset_index(drop=True)
    return df_final

In [16]:
dataset_final_hibrido = pipeline_vae_personalizado('C:\\Users\\Usuario\\Documents\\Workspace\\Mirage\\dataset_bert_undersampled.csv', 'C:\\Users\\Usuario\\Documents\\Workspace\\Mirage\\code\\tramo_final\\config_aumento.csv') 
dataset_final_hibrido.to_csv('embeddings_entrenamiento_vae.csv', index=False)


>>> Procesando F20 (Reales: 600)
    Saltando aumento para F20 (Porcentaje 0).

>>> Procesando F25 (Reales: 110)
    Generando 38 sintéticos (Aumento del 35.0%)...

>>> Procesando F29 (Reales: 149)
    Generando 52 sintéticos (Aumento del 35.0%)...

>>> Procesando F22 (Reales: 270)
    Generando 94 sintéticos (Aumento del 35.0%)...

>>> Procesando F23 (Reales: 70)
    Generando 24 sintéticos (Aumento del 35.0%)...

>>> Procesando F60.1 (Reales: 13)
    Generando 26 sintéticos (Aumento del 200.0%)...

>>> Procesando F21 (Reales: 7)
    Generando 14 sintéticos (Aumento del 200.0%)...
